In [1]:
import sys
print(sys.executable)

c:\Users\Tehila\AppData\Local\Programs\Python\Python313\python.exe


In [2]:
import gzip
import json
import os
import time

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [3]:
INPUT_PATH = r"G:\My Drive\git-archive-huge.json.gz"  # לשנות אם שם הקובץ אצלכן אחר
OUTPUT_PATH = "events.parquet"
BATCH_SIZE = 100_000

B1 - הגדרת הסכמה

In [13]:
schema = pa.schema([
    ("event_id", pa.string()),
    ("event_type", pa.dictionary(pa.int8(), pa.string())),
    ("actor_login", pa.string()),
    ("repo_name", pa.string()),
    ("created_at", pa.timestamp("us", tz="UTC")),
    ("commit_count", pa.int32()),
])

print(schema)

event_id: string
event_type: dictionary<values=string, indices=int8, ordered=0>
actor_login: string
repo_name: string
created_at: timestamp[us, tz=UTC]
commit_count: int32


B2 - פונקציית flatten

In [5]:
def flatten_event(raw: dict) -> dict:
    payload = raw.get("payload") or {}
    actor = raw.get("actor") or {}
    repo = raw.get("repo") or {}

    return {
        "event_id": str(raw.get("id") or ""),
        "event_type": raw.get("type") or "",
        "actor_login": actor.get("login") or "",
        "repo_name": repo.get("name") or "",
        "created_at": raw.get("created_at") or "",
        "commit_count": int(payload.get("size") or 0),
    }

In [6]:
test = {
    "id": "123",
    "type": "PushEvent",
    "actor": {"login": "tehila"},
    "repo": {"name": "owner/repo"},
    "created_at": "2015-01-01T10:00:00Z",
    "payload": {"size": 3}
}

flatten_event(test)

{'event_id': '123',
 'event_type': 'PushEvent',
 'actor_login': 'tehila',
 'repo_name': 'owner/repo',
 'created_at': '2015-01-01T10:00:00Z',
 'commit_count': 3}

B3 - המרה לparquet 

In [7]:
import os

print(os.getcwd())
print(os.listdir())

c:\Users\Tehila\OneDrive - Jerusalem College of Technology - Machon Lev\מכון טל\שנה ד\big data\Targil-GraphColStore
['.git', '.venv', '01_explore.ipynb', '02_to_parquet.ipynb', 'events.parquet', 'README.md']


In [8]:
import os

if os.path.exists("events.parquet"):
    os.remove("events.parquet")
    print("deleted partial events.parquet")

start = time.perf_counter()

rows = []
total_rows = 0

writer = pq.ParquetWriter(
    OUTPUT_PATH,
    schema=schema,
    compression="snappy"
)

with gzip.open(INPUT_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        raw = json.loads(line)
        rows.append(flatten_event(raw))

        if len(rows) >= BATCH_SIZE:
            table = pa.Table.from_pylist(rows, schema=schema)
            writer.write_table(table)
            total_rows += len(rows)

            if total_rows % 1_000_000 == 0:
                print(f"{total_rows:,} rows written")
            
            rows = []

    if rows:
        table = pa.Table.from_pylist(rows, schema=schema)
        writer.write_table(table)
        total_rows += len(rows)

writer.close()

conversion_time = time.perf_counter() - start
throughput = total_rows / conversion_time

print(f"Total rows: {total_rows:,}")
print(f"Conversion time: {conversion_time:.2f} seconds")
print(f"Throughput: {throughput:,.2f} rows/sec")

deleted partial events.parquet
1,000,000 rows written
2,000,000 rows written
3,000,000 rows written
4,000,000 rows written
5,000,000 rows written
6,000,000 rows written
7,000,000 rows written
8,000,000 rows written
9,000,000 rows written
10,000,000 rows written
11,000,000 rows written
12,000,000 rows written
13,000,000 rows written
14,000,000 rows written
15,000,000 rows written
16,000,000 rows written
17,000,000 rows written
18,000,000 rows written
19,000,000 rows written
20,000,000 rows written
21,000,000 rows written
22,000,000 rows written
23,000,000 rows written
24,000,000 rows written
25,000,000 rows written
26,000,000 rows written
27,000,000 rows written
28,000,000 rows written
Total rows: 28,506,909
Conversion time: 6751.00 seconds
Throughput: 4,222.62 rows/sec


B4 - אימות הפלט

In [9]:
start = time.perf_counter()

print(pq.read_schema(OUTPUT_PATH))

df_500 = pq.read_table(OUTPUT_PATH).slice(0, 500).to_pandas()

reload_time = time.perf_counter() - start

df_500.head()

event_id: string
event_type: string
actor_login: string
repo_name: string
created_at: string
commit_count: int32


,event_id,event_type,actor_login,repo_name,created_at,commit_count
0,2594237218,IssueCommentEvent,yongli-d,yongli-d/StravaBuddy,2015-02-20T01:00:00Z,0
1,2594237220,CreateEvent,vgeshel,yummly/s3-to-redshift,2015-02-20T01:00:01Z,0
2,2594237222,PushEvent,davidcarlsonberg,PubWlkr/PubWlkr,2015-02-20T01:00:01Z,1
3,2594237223,IssueCommentEvent,wronk,mne-tools/mne-python,2015-02-20T01:00:01Z,0
4,2594237233,WatchEvent,mcstiches,raywenderlich/swift-style-guide,2015-02-20T01:00:01Z,0


In [10]:
original_size_mb = os.path.getsize(INPUT_PATH) / (1024 ** 2)
parquet_size_mb = os.path.getsize(OUTPUT_PATH) / (1024 ** 2)
ratio = parquet_size_mb / original_size_mb

memory_mb = df_500.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"Original JSON.GZ size: {original_size_mb:.2f} MB")
print(f"Parquet size: {parquet_size_mb:.2f} MB")
print(f"Conversion ratio: {ratio:.3f}")
print(f"DataFrame memory usage: {memory_mb:.2f} MB")
print(f"Reload time: {reload_time:.2f} seconds")

Original JSON.GZ size: 10180.67 MB
Parquet size: 746.15 MB
Conversion ratio: 0.073
DataFrame memory usage: 0.06 MB
Reload time: 63.10 seconds
